# Автоэнкодеры

**AI2 · Computer Vision** — [AI Olympiad Curriculum](https://ai-olympiad-curriculum.netlify.app)

~15 min

## Введение

При обучении CNN одна из проблем заключается в том, что требуется большое количество размеченных данных. В случае классификации изображений необходимо разделить изображения на разные классы, что требует ручной работы.


Однако мы можем использовать необработанные (неразмеченные) данные для обучения CNN в качестве извлекателей признаков, что называется **самообучением** (self-supervised). Вместо меток мы будем использовать обучающие изображения как входные данные и как выходные. Основная идея **автоэнкодера** заключается в том, что у нас будет **кодирующая сеть**, которая преобразует входное изображение в некоторое **латентное пространство** (обычно это просто вектор меньшего размера), а затем **декодирующая сеть**, цель которой — восстановить исходное изображение.

> ✅ [Автоэнкодер](https://wikipedia.org/wiki/Autoencoder) — это "тип искусственной нейронной сети, используемой для изучения эффективного кодирования неразмеченных данных."

Поскольку мы обучаем автоэнкодер захватывать как можно больше информации из исходного изображения для точного восстановления, сеть пытается найти наилучшее **встраивание** входных изображений, чтобы уловить их смысл.

![Схема автоэнкодера](ms-assets/ai/autoencoder_schema.5e6fc9ad98a5eb61.webp)

> Изображение из [блога Keras](https://blog.keras.io/building-autoencoders-in-keras.html)



## Сценарии использования автоэнкодеров

Хотя восстановление исходных изображений само по себе может показаться не слишком полезным, есть несколько сценариев, где автоэнкодеры особенно эффективны:

* **Снижение размерности изображений для визуализации** или **обучение встраиванию изображений**. Обычно автоэнкодеры дают лучшие результаты, чем PCA, поскольку учитывают пространственную природу изображений и иерархические признаки.
* **Удаление шума**, то есть избавление изображения от шума. Поскольку шум содержит много бесполезной информации, автоэнкодер не может вместить его весь в относительно небольшое латентное пространство, и поэтому захватывает только важную часть изображения. При обучении автоэнкодеров для удаления шума мы начинаем с исходных изображений и используем изображения с искусственно добавленным шумом в качестве входных данных.
* **Сверхразрешение**, увеличение разрешения изображения. Мы начинаем с изображений высокого разрешения и используем изображение с более низким разрешением в качестве входных данных для автоэнкодера.
* **Генеративные модели**. После обучения автоэнкодера его декодирующую часть можно использовать для создания новых объектов, начиная с случайных латентных векторов.



## Вариационные автоэнкодеры (VAE)

Традиционные автоэнкодеры уменьшают размерность входных данных, выделяя важные признаки входных изображений. Однако латентные векторы часто не имеют очевидного смысла. Например, если взять набор данных MNIST, определить, какие цифры соответствуют различным латентным векторам, может быть непросто, поскольку близкие латентные векторы не обязательно будут соответствовать одним и тем же цифрам.

С другой стороны, для обучения *генеративных* моделей лучше иметь некоторое представление о латентном пространстве. Эта идея приводит нас к **вариационному автоэнкодеру** (VAE).

VAE — это автоэнкодер, который учится предсказывать *статистическое распределение* латентных параметров, так называемое **латентное распределение**. Например, мы можем захотеть, чтобы латентные векторы имели нормальное распределение с некоторым средним z<sub>mean</sub> и стандартным отклонением z<sub>sigma</sub> (оба — векторы некоторой размерности d). Кодировщик в VAE учится предсказывать эти параметры, а затем декодер берет случайный вектор из этого распределения для восстановления объекта.

Итак, подытожим:

 * Из входного вектора мы предсказываем `z_mean` и `z_log_sigma` (вместо предсказания самого стандартного отклонения мы предсказываем его логарифм)
 * Мы выбираем случайный вектор `sample` из распределения N(z<sub>mean</sub>,exp(z<sub>log\_sigma</sub>))
 * Декодер пытается декодировать исходное изображение, используя `sample` в качестве входного вектора

 ![](ms-assets/ai/vae.464c465a5b6a9e25.webp)

> Изображение из [этого блога](https://ijdykeman.github.io/ml/2016/12/21/cvae.html) автора Исаака Дайкмана

Вариационные автоэнкодеры используют сложную функцию потерь, состоящую из двух частей:

* **Потери восстановления** — это функция потерь, показывающая, насколько близко восстановленное изображение к целевому (может быть среднеквадратичная ошибка, или MSE). Это та же функция потерь, что и в обычных автоэнкодерах.
* **KL-потери**, которые обеспечивают, чтобы распределение латентных переменных оставалось близким к нормальному распределению. Они основаны на понятии [дивергенции Кульбака-Лейблера](https://www.countbayesie.com/blog/2017/5/9/kullback-leibler-divergence-explained) — метрике для оценки сходства двух статистических распределений.

Одно из важных преимуществ VAE заключается в том, что они позволяют относительно легко генерировать новые изображения, поскольку мы знаем, из какого распределения выбирать латентные векторы. Например, если мы обучим VAE с 2D латентным вектором на MNIST, мы сможем изменять компоненты латентного вектора, чтобы получать разные цифры:

![vaemnist](ms-assets/ai/vaemnist.cab9e602dc08dc50.webp)

> Изображение от [Дмитрия Сошникова](http://soshnikov.com)

Обратите внимание, как изображения плавно переходят друг в друга, когда мы начинаем брать латентные векторы из разных областей латентного пространства. Мы также можем визуализировать это пространство в 2D:

![vaemnist cluster](ms-assets/ai/vaemnist-diag.694315f775d5d666.webp) 

> Изображение от [Дмитрия Сошникова](http://soshnikov.com)



## Упражнения: Автоэнкодеры

Узнайте больше об автоэнкодерах в этих соответствующих ноутбуках:

* [Автоэнкодеры в TensorFlow](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/4-ComputerVision/09-Autoencoders/AutoencodersTF.ipynb)
* [Автоэнкодеры в PyTorch](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/4-ComputerVision/09-Autoencoders/AutoEncodersPyTorch.ipynb)



## Свойства автоэнкодеров

* **Специфичность данных** — они хорошо работают только с тем типом изображений, на которых были обучены. Например, если мы обучим сеть сверхразрешения на цветах, она будет плохо работать с портретами. Это связано с тем, что сеть может создавать изображения с более высоким разрешением, используя тонкие детали, извлеченные из признаков, изученных на обучающем наборе данных.
* **С потерями (lossy)** — восстановленное изображение не идентично исходному. Характер потерь определяется *функцией потерь*, используемой во время обучения.
* Работают с **неразмеченными данными**



## Заключение

В этом уроке вы узнали о различных типах автоэнкодеров, доступных для исследователя ИИ. Вы узнали, как их строить и использовать для восстановления изображений. Вы также узнали о VAE и о том, как использовать его для генерации новых изображений.



## Задание

В этом уроке вы узнали об использовании автоэнкодеров для изображений. Но их также можно использовать для музыки! Ознакомьтесь с проектом Magenta [MusicVAE](https://magenta.tensorflow.org/music-vae), который использует автоэнкодеры для обучения восстановления музыки. Проведите [эксперименты](https://colab.research.google.com/github/magenta/magenta-demos/blob/master/colab-notebooks/Multitrack_MusicVAE.ipynb) с этой библиотекой, чтобы увидеть, что вы можете создать.



## Обзор и самостоятельное изучение

Для справки, прочитайте больше об автоэнкодерах в этих ресурсах:

* [Создание автоэнкодеров в Keras](https://blog.keras.io/building-autoencoders-in-keras.html)
* [Статья в блоге NeuroHive](https://neurohive.io/ru/osnovy-data-science/variacionnyj-avtojenkoder-vae/)
* [Объяснение вариационных автоэнкодеров](https://kvfrans.com/variational-autoencoders-explained/)
* [Условные вариационные автоэнкодеры](https://ijdykeman.github.io/ml/2016/12/21/cvae.html)



## Задание

В конце [этого ноутбука с TensorFlow](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/4-ComputerVision/09-Autoencoders/AutoencodersTF.ipynb) вы найдете "задачу" — используйте ее в качестве вашего задания.



---

## ✅ Self-check tasks

Implement the function in the 'your code' cell, then run the check cell — it runs randomized tests and shows the first mismatch. Run the helper cell below once first.

In [ ]:
# ── task-checking helper cell — run once ──
import copy
import numpy as np
import math, statistics, itertools, collections, functools, heapq, random

try:
    import sys; sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

_L = {"noFn":"❓ Define a function `{e}` in the solution cell and run it","runErr":"❌ Test {i}: {t}: {m}","wrong":"❌ Test {i}: {r}","badRef":"⚠️ Check harness broke (generator/reference): {t}: {m}","inL":"Input","expL":"Expected","gotL":"Got","ok":"✅ Correct! All {n} tests passed."}

def _nb_copy_kw(kw):
    return {k: (v.copy() if isinstance(v, np.ndarray) else copy.deepcopy(v)) for k, v in kw.items()}

def _nb_jsafe(x, depth=0):
    if depth > 6: return "…"
    if isinstance(x, np.ndarray): return _nb_jsafe(x.tolist(), depth + 1)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, float): return round(x, 6) if math.isfinite(x) else str(x)
    if isinstance(x, (list, tuple)): return [_nb_jsafe(v, depth + 1) for v in x]
    if isinstance(x, dict): return {str(k): _nb_jsafe(v, depth + 1) for k, v in x.items()}
    if isinstance(x, (int, str, bool)) or x is None: return x
    return repr(x)

def _nb_cmp(exp, got, path=""):
    if isinstance(exp, np.ndarray) or isinstance(got, np.ndarray):
        e, g = np.asarray(exp), np.asarray(got)
        if e.shape != g.shape:
            return f"shape {tuple(g.shape)} != expected {tuple(e.shape)}"
        try:
            if np.issubdtype(e.dtype, np.number) or np.issubdtype(g.dtype, np.number):
                ef, gf = e.astype(float), g.astype(float)
                close = np.isclose(gf, ef, atol=1e-6, rtol=1e-5, equal_nan=True)
                if not bool(close.all()):
                    i = tuple(int(v) for v in np.argwhere(~close)[0])
                    return f"{path}first mismatch at index {i}: got {gf[i]:.6g} vs expected {ef[i]:.6g}"
            elif not np.array_equal(g, e):
                return f"{path}values differ"
        except Exception as ex:
            return f"{path}cannot compare: {ex}"
        return None
    if isinstance(exp, (int, float)) and not isinstance(exp, bool) and isinstance(got, (int, float)) and not isinstance(got, bool):
        if not np.isclose(float(exp), float(got), atol=1e-6, rtol=1e-5):
            return f"{path}{got} != expected {exp}"
        return None
    if isinstance(exp, (list, tuple)) and isinstance(got, (list, tuple)):
        if len(exp) != len(got):
            return f"{path}length {len(got)} != expected {len(exp)}"
        for i, (a, b) in enumerate(zip(exp, got)):
            r = _nb_cmp(a, b, path + f"[{i}].")
            if r: return r
        return None
    if isinstance(exp, dict) and isinstance(got, dict):
        if set(exp) != set(got):
            return f"{path}keys differ"
        for k in exp:
            r = _nb_cmp(exp[k], got[k], path + f"[{k!r}].")
            if r: return r
        return None
    if exp != got:
        return f"{path}{got!r} != expected {exp!r}"
    return None

def _check_task(g, ref_src, entry, n=20, seed=0):
    """g — globals() ноутбука; ref_src — gen() + эталонное решение."""
    fn = g.get(entry)
    if not callable(fn):
        print(_L["noFn"].format(e=entry)); return
    ns = {"np": np, "math": math, "statistics": statistics, "itertools": itertools,
          "collections": collections, "functools": functools, "heapq": heapq, "random": random}
    try:
        exec(ref_src, ns)
        gen, ref = ns["gen"], ns[entry]
    except Exception as e:
        print(_L["badRef"].format(t=type(e).__name__, m=e)); return
    rng = np.random.default_rng(seed)
    for i in range(n):
        try:
            kw = gen(rng)
        except Exception as e:
            print(_L["badRef"].format(t=type(e).__name__, m=e)); return
        try:
            got = fn(**_nb_copy_kw(kw))
        except Exception as e:
            print(_L["runErr"].format(i=i + 1, t=type(e).__name__, m=e)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))); return
        exp = ref(**_nb_copy_kw(kw))
        r = _nb_cmp(exp, got)
        if r:
            print(_L["wrong"].format(i=i + 1, r=r)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))
                  + "\n" + _L["expL"] + ": " + repr(_nb_jsafe(exp))
                  + "\n" + _L["gotL"] + ": " + repr(_nb_jsafe(got))); return
    print(_L["ok"].format(n=n))

### Task 1 — Reparameterization trick

●○○ · ~6 min

> Sampling z ~ N(μ, σ²) is not differentiable; the reparameterization trick writes z = μ + σ·ε with ε ~ N(0, I), moving randomness into an input so gradients flow through μ and σ. VAEs store log-variance rather than σ: σ = exp(0.5·logvar) is guaranteed positive.

Implement `reparameterize(mu, logvar, eps)` → `mu + exp(0.5 * logvar) * eps`. All inputs are same-shape arrays; `eps` is the pre-drawn standard normal noise.

<details><summary>Hint</summary>

sigma = np.exp(0.5 * logvar); then mu + sigma * eps.

</details>


In [ ]:
import numpy as np

def reparameterize(mu, logvar, eps):
    mu = np.asarray(mu, dtype=float)
    # your code
    return mu

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    d = int(rng.integers(2, 9))\n    return {\"mu\": rng.normal(0, 1, d),\n            \"logvar\": rng.normal(-1, 1.0, d),\n            \"eps\": rng.normal(0, 1, d)}\n\ndef reparameterize(mu, logvar, eps):\n    mu = np.asarray(mu, dtype=float)\n    logvar = np.asarray(logvar, dtype=float)\n    eps = np.asarray(eps, dtype=float)\n    return mu + np.exp(0.5 * logvar) * eps", "reparameterize", 20)

### Task 2 — KL between diagonal Gaussians

●●○ · ~10 min

> The VAE objective pulls the encoder posterior q toward the prior p through KL(N_q || N_p), which for diagonal Gaussians has the closed form 0.5·Σ(σ_q²/σ_p² + (μ_p − μ_q)²/σ_p² − 1 + ln σ_p² − ln σ_q²). Working in log-variance keeps it stable: variance ratios become exp(logvar_q − logvar_p), so you never explicitly divide by a tiny σ².

Implement `kl_diag(mu_q, logvar_q, mu_p, logvar_p)` → scalar KL(N(μ_q, diag e^{logvar_q}) || N(μ_p, diag e^{logvar_p})) = `0.5 * sum(exp(logvar_q - logvar_p) + (mu_p - mu_q)^2 * exp(-logvar_p) - 1 + logvar_p - logvar_q)`.

<details><summary>Hint</summary>

term = np.exp(logvar_q - logvar_p) + (mu_p - mu_q)**2 * np.exp(-logvar_p) - 1.0 + logvar_p - logvar_q; return 0.5 * term.sum().

</details>


In [ ]:
import numpy as np

def kl_diag(mu_q, logvar_q, mu_p, logvar_p):
    mu_q = np.asarray(mu_q, dtype=float)
    # your code
    return 0.0

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    d = int(rng.integers(2, 9))\n    return {\"mu_q\": rng.normal(0, 1, d), \"logvar_q\": rng.normal(0, 1, d),\n            \"mu_p\": rng.normal(0, 1, d), \"logvar_p\": rng.normal(0, 1, d)}\n\ndef kl_diag(mu_q, logvar_q, mu_p, logvar_p):\n    mu_q = np.asarray(mu_q, dtype=float)\n    logvar_q = np.asarray(logvar_q, dtype=float)\n    mu_p = np.asarray(mu_p, dtype=float)\n    logvar_p = np.asarray(logvar_p, dtype=float)\n    term = np.exp(logvar_q - logvar_p) + (mu_p - mu_q) ** 2 * np.exp(-logvar_p) - 1.0 + logvar_p - logvar_q\n    return float(0.5 * term.sum())", "kl_diag", 20)

### Task 3 — PSNR between images

●●○ · ~10 min

> Restoration quality is measured in decibels: PSNR = 10·log10(MAX² / MSE), where MAX is the peak signal value (1.0 for images normalized to [0, 1]) and MSE the mean squared error between restored and ground-truth images. Higher is better; identical images give +∞ — a case worth handling explicitly.

Implement `psnr(a, b)` for two same-shape float images in [0, 1] → `10·log10(1 / mse)` (MAX = 1, so MAX² = 1). If `a` and `b` are identical (mse == 0), return `float('inf')`.

<details><summary>Hint</summary>

mse = float(((a − b) ** 2).mean()); guard mse == 0 → inf; otherwise 10 * np.log10(1 / mse).

</details>


In [ ]:
import numpy as np

def psnr(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    # your code
    return 0.0

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    a = rng.random((int(rng.integers(3, 10)), int(rng.integers(3, 10)), 3))\n    b = np.clip(a + rng.normal(0, 0.15, a.shape), 0.0, 1.0)\n    if rng.random() < 0.15:\n        b = a.copy()          # identical → mse 0 → spec requires float('inf')\n    return {\"a\": a, \"b\": b}\n\ndef psnr(a, b):\n    a = np.asarray(a, dtype=float)\n    b = np.asarray(b, dtype=float)\n    mse = float(((a - b) ** 2).mean())\n    if mse == 0.0:\n        return float(\"inf\")\n    return float(10.0 * np.log10(1.0 / mse))", "psnr", 30)

---

**Go deeper**
- [Исходный урок (Microsoft, MIT)](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/4-ComputerVision/09-Autoencoders)
- [Учебник ML (ШАД): свёрточные нейросети](https://contest.yandex.ru/tracks/ml/deep-learning-architectures/convolutional-neural-networks)
- [Учебник ML (ШАД): генеративные модели](https://contest.yandex.ru/tracks/ml/generative-models/introduction-to-generative-modeling)